In [1]:
import torch 
import os


/home/peppe/miniconda3/envs/GNRR/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
dataset_name = 'BM25_1300_K_8_TENS_DIM_F_64' # 'bm25_1000_k_64_np32' / 'Oracolar_BM25_1000_K_64_TENS_DIM_F_64'

def retrieve_dataset_from_file(dataset_name):
    dataset = []
    path = f'../data/{dataset_name}/tensors/'
    list_of_files = os.listdir(path)
    print(len(list_of_files))
    for file in list_of_files:
        print(file)
        diz = {}
        adj_matr = torch.load(path + file + '/adjacency_matrix.pt').type(torch.FloatTensor)
        doc_feat = torch.load(path + file + '/doc_feat_tensor.pt')
        qrels_tensor = torch.load(path + file + '/qrels_tensor.pt')
        query_tensor = torch.load(path + file + '/query_tensor.pt')
        qid = int(file[4])
        diz['doc_feat'] = doc_feat
        diz['adj_matrix'] = adj_matr
        diz['q_ids'] = qid
        diz['q_rels'] = qrels_tensor
        diz['query_feat'] = query_tensor





        dataset.append(diz)
    return dataset


In [10]:
data_normal = retrieve_dataset_from_file(dataset_name)


50
qid_7_tensors
qid_8_tensors
qid_38_tensors
qid_35_tensors
qid_26_tensors
qid_11_tensors
qid_25_tensors
qid_30_tensors
qid_29_tensors
qid_34_tensors
qid_18_tensors
qid_44_tensors
qid_42_tensors
qid_12_tensors
qid_50_tensors
qid_47_tensors
qid_48_tensors
qid_1_tensors
qid_46_tensors
qid_27_tensors
qid_45_tensors
qid_16_tensors
qid_19_tensors
qid_14_tensors
qid_31_tensors
qid_37_tensors
qid_5_tensors
qid_36_tensors
qid_10_tensors
qid_3_tensors
qid_17_tensors
qid_24_tensors
qid_23_tensors
qid_49_tensors
qid_6_tensors
qid_4_tensors
qid_41_tensors
qid_32_tensors
qid_15_tensors
qid_9_tensors
qid_33_tensors
qid_40_tensors
qid_2_tensors
qid_21_tensors
qid_22_tensors
qid_13_tensors
qid_39_tensors
qid_28_tensors
qid_20_tensors
qid_43_tensors


In [8]:
data_normal[0]['adj_matrix'].shape

torch.Size([2, 5114])

In [7]:
import torch
from torch_geometric.data import Data
from torch_geometric.utils import degree

def count_isolated_nodes_coo(coo_tensor):
    # Convert COO tensor to a PyTorch Geometric Data object
    edge_index = coo_tensor
    data = Data(edge_index=edge_index)
   
    # Calculate the degree of each node
    degrees = degree(data.edge_index[0, :].type(torch.LongTensor), data.num_nodes)

    # Find isolated nodes (degree == 0)
    isolated_nodes = (degrees == 0).nonzero(as_tuple=False).squeeze()

    # Count the number of isolated nodes
    num_isolated_nodes = isolated_nodes.size(0)

    return num_isolated_nodes

count_isolated_nodes_coo(data_normal[0]['adj_matrix'])

/home/peppe/miniconda3/envs/GNRR/lib/python3.8/site-packages/torch_geometric/data/storage.py:304: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  warnings.warn(


114

In [ ]:
import pandas as pd

columns = ['Delta_Connections', 'Delta_qrels', 'Delta_Isolated']

dataframe = pd.DataFrame(columns = columns)
    # print(data_oracle[i]['q_rels'])
dataframe.rows = list(range(len(data_normal)))

for i in range(len(data_normal)):
    delta_connection = data_oracle[i]['adj_matrix'].shape[-1] - data_normal[i]['adj_matrix'].shape[-1] 
    delta_qrels = torch.sum((data_oracle[i]['q_rels'] == 2) | (data_oracle[i]['q_rels'] == 1)) - torch.sum((data_normal[i]['q_rels'] == 2) | (data_normal[i]['q_rels'] == 1)) 
    delta_isolated = count_isolated_nodes_coo(data_oracle[i]['adj_matrix']) - count_isolated_nodes_coo(data_normal[i]['adj_matrix'])


    dataframe.loc[i, 'Delta_Connections'] = delta_connection
    dataframe.loc[i, 'Delta_qrels'] = delta_qrels.item()
    dataframe.loc[i, 'Delta_Isolated'] = delta_isolated




display(dataframe)



In [ ]:
import torch
import networkx as nx
import matplotlib.pyplot as plt

def coo_to_adjacency_matrix(coo_tensor):
    row_indices = coo_tensor[0].long()
    col_indices = coo_tensor[1].long()
    size = (max(row_indices.max(), col_indices.max()) + 1, max(row_indices.max(), col_indices.max()) + 1)
    # Create a sparse COO tensor
    sparse_coo = torch.sparse_coo_tensor(indices=torch.stack([row_indices, col_indices]),
                                         values=torch.ones_like(row_indices).float(),
                                         size=size)
    # Convert the sparse COO tensor to a dense adjacency matrix
    adjacency_matrix = sparse_coo.to_dense()
    return adjacency_matrix


def plot_graph_from_adjacency(adjacency_tensor, directed=False):
    # Step 1: Convert the adjacency tensor to an adjacency matrix
    adjacency_matrix = coo_to_adjacency_matrix(adjacency_tensor)

    # Step 2: Identify isolated nodes
    isolated_nodes = torch.all(adjacency_matrix == 0, dim=0) & torch.all(adjacency_matrix == 0, dim=1)

    # Step 3: Remove isolated nodes from the adjacency matrix
    adjacency_matrix = adjacency_matrix#[~isolated_nodes][:, ~isolated_nodes]

    # Step 4: Create a graph using NetworkX
    if directed:
        G = nx.from_numpy_array(adjacency_matrix.numpy(), create_using=nx.DiGraph)
    else:
        G = nx.from_numpy_array(adjacency_matrix.numpy())

    # Step 5: Plot the graph with improved settings
    pos = nx.spring_layout(G, seed=42)  # Using spring_layout with a seed for reproducibility

    # Customize node colors, sizes, labels, etc., based on your requirements
    fig, ax = plt.subplots(figsize=(200, 100))  # Adjust figsize as needed
    nx.draw(G, pos, with_labels=True, node_color='skyblue', node_size=400, font_size=8, font_color='black', font_weight='bold', arrowsize=10 if directed else 0, ax=ax)

    plt.title("Graph Visualization")
    plt.show()

for i in range(len(data_normal)):
    edges_tensor = data_normal[i]['adj_matrix']
    plot_graph_from_adjacency(edges_tensor, directed=True)
    if i == 0:
        break




In [ ]:
data_normal[0]['doc_feat'].shape


In [ ]:
def cosine_similarity_row_by_row(matrix1, matrix2):
    dot_product = torch.sum(matrix1 * matrix2, dim=1)
    norm_matrix1 = torch.norm(matrix1, dim=1)
    norm_matrix2 = torch.norm(matrix2, dim=1)

    similarity = dot_product / (norm_matrix1 * norm_matrix2 + 1e-8)  # Adding a small epsilon to avoid division by zero

    return similarity

out = cosine_similarity_row_by_row(x_reshaped, query_reshaped)

In [ ]:
from torchmetrics.functional.retrieval import retrieval_normalized_dcg


preds = torch.tensor([-0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.2663,
                     -0.2766,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607,
                     -0.3607], device='cuda:0')

targets = torch.tensor([[2.],
                        [1.],
                        [1.],
                        [1.],
                        [2.],
                        [2.],
                        [1.],
                        [2.],
                        [2.],
                        [2.],
                        [1.],
                        [1.],
                        [2.],
                        [2.],
                        [2.],
                        [1.],
                        [2.],
                        [1.],
                        [1.],
                        [1.]])

# Flatten into one-dimensional vectors
preds = preds.view(-1)
targets = targets.view(-1)



ndcg = retrieval_normalized_dcg(preds, targets)
# You can use flat_preds and flat_targets as one-dimensional vectors in your code.


In [ ]:
import torch.nn.functional as F
import torch.nn as nn

class ListNetLoss(nn.Module):

    def __init__(self):
        super(ListNetLoss, self).__init__()

    def forward(self, y_pred, y_true, eps=1e-15):
        """
        ListNet loss introduced in "Learning to Rank: From Pairwise Approach to Listwise Approach".
        :param y_pred: predictions from the model, shape [batch_size, slate_length]
        :param y_true: ground truth labels, shape [batch_size, slate_length]
        :return: loss value, a torch.Tensor
        """
        y_pred = y_pred.clone()
        y_true = y_true.clone()

        preds_smax = F.softmax(y_pred, dim = 1)
        true_smax = F.softmax(y_true, dim = 1)

        preds_smax = preds_smax + eps
        preds_log = torch.log(preds_smax)

        return torch.mean(-torch.sum(true_smax * preds_log, dim=1))
    

class ListMLELoss(nn.Module):

    def __init__(self):
        super(ListMLELoss, self).__init__()

    def forward(self, y_pred, y_true, eps=1e-15, padded_value_indicator=-1.):
        
        """
        ListMLE loss introduced in "Listwise Approach to Learning to Rank - Theory and Algorithm".
        :param y_pred: predictions from the model, shape [batch_size, slate_length]
        :param y_true: ground truth labels, shape [batch_size, slate_length]
        :return: loss value, a torch.Tensor
        """

        y_pred = y_pred.clone().squeeze(-1).unsqueeze(0)
        y_true = y_true.clone().squeeze(-1).unsqueeze(0)

        print(y_pred.shape)
        print(y_true.shape)


        # shuffle for randomised tie resolution
        random_indices = torch.randperm(y_pred.shape[-1])
        y_pred_shuffled = y_pred[:, random_indices]
        y_true_shuffled = y_true[:, random_indices]

        y_true_sorted, indices = y_true_shuffled.sort(descending=True, dim=-1)

        mask = y_true_sorted == padded_value_indicator

        preds_sorted_by_true = torch.gather(y_pred_shuffled, dim=1, index=indices)
        preds_sorted_by_true[mask] = float("-inf")

        max_pred_values, _ = preds_sorted_by_true.max(dim=1, keepdim=True)

        preds_sorted_by_true_minus_max = preds_sorted_by_true - max_pred_values

        cumsums = torch.cumsum(preds_sorted_by_true_minus_max.exp().flip(dims=[1]), dim=1).flip(dims=[1])

        observation_loss = torch.log(cumsums + eps) - preds_sorted_by_true_minus_max

        observation_loss[mask] = 0.0

        return torch.mean(torch.sum(observation_loss, dim=1))




In [ ]:

preds = torch.tensor([10., 8., 4., 8., 8., 7., 4.])
target = torch.tensor([2., 2., 1., 2., 2., 2., 1.])

loss = ListMLELoss()

In [ ]:
loss(preds, target)